In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np

from util import Pipeline, load_base_year_emp
from util.elmer_helpers import read_from_elmer_geo, read_from_elmer

p = Pipeline('configs')

In [16]:
def gdf_to_gpkg(pipeline,name):
    p = pipeline
    gdf = p.get_geodataframe(name)
    gdf.to_file(f'data/{name}.gpkg', driver='GPKG')

In [17]:
gdf_to_gpkg(p,'regional_geographies')
gdf_to_gpkg(p,'county')

In [130]:
reg = p.get_geodataframe('regional_geographies')
reg = reg.reset_index(drop=True)
reg['reg_id'] = reg.index + 1
county = p.get_geodataframe('county').query("psrc == 1")

included_milspn_ids = [12,13,19,20,21,22]
military = p.get_geodataframe('military_bases').query("milspn_id in @included_milspn_ids").dissolve('milspn_id').reset_index(drop=True)
military['military_id'] = military.index + 1

In [140]:
def union_snap_boundary(input_gdf, input_gdf_id, starting_gdf, starting_gdf_id, tolerance):
    input_gdf = gpd.clip(input_gdf, starting_gdf.dissolve())
    m_buffer = input_gdf[[input_gdf_id,'geometry']].copy()
    m_buffer['geometry'] = m_buffer.geometry.buffer(tolerance)

    overlay = gpd.overlay(input_gdf, starting_gdf, how='union')
    overlay = overlay.explode().reset_index(drop=True)
    overlay['exp_id'] = overlay.index

    overlay_pts = overlay.copy()
    overlay_pts['geometry'] = overlay_pts.representative_point()

    overlay_pts = overlay_pts[['exp_id','geometry']].sjoin(m_buffer)
    overlay_pts = overlay_pts.rename(columns={input_gdf_id:f'{input_gdf_id}_buffer'})
    overlay = overlay.merge(overlay_pts[['exp_id',f'{input_gdf_id}_buffer']], on='exp_id',how='left')
    overlay = overlay.dissolve([f'{input_gdf_id}_buffer',starting_gdf_id],dropna=False)
    return overlay

In [132]:
cnty_mil = union_snap_boundary(military, 'military_id',county, 'county_fip', 150)
cnty_mil.to_file('data/cnty_mil.gpkg', driver='GPKG')

In [133]:
cnty_mil = cnty_mil.reset_index()
cnty_mil['cnty_mil_id'] = cnty_mil.index + 1
cnty_mil = cnty_mil[['cnty_mil_id','name','county_fip','geometry']]

In [129]:
cnty_mil.to_file('data/cnty_mil.gpkg', driver='GPKG')

In [142]:
cnty_mil.overlay(reg).to_file('data/cnty_mil_reg.gpkg', driver='GPKG')

In [141]:
cnty_mil_reg = union_snap_boundary(reg, 'reg_id',cnty_mil, 'cnty_mil_id', 150)
cnty_mil_reg.to_file('data/cnty_mil_reg.gpkg', driver='GPKG')

c:\Users\JKolberg\PythonProjects\control_totals\.venv\Lib\site-packages\geopandas\tools\overlay.py:358: UserWarning: `keep_geom_type=True` in overlay resulted in 50 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  result = _collection_extract(result, geom_type, keep_geom_type_warning)


In [145]:
reg_mil = union_snap_boundary(military, 'military_id',reg, 'reg_id', 500)
reg_mil.to_file('data/reg_mil.gpkg', driver='GPKG')

c:\Users\JKolberg\PythonProjects\control_totals\.venv\Lib\site-packages\geopandas\tools\overlay.py:358: UserWarning: `keep_geom_type=True` in overlay resulted in 26 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  result = _collection_extract(result, geom_type, keep_geom_type_warning)
